# Hearo - YAMNet Fine-Tuning
청각장애인을 위한 가정용 환경음 인식 모델 파인튜닝

## 6개 분류 카테고리
1. 개 짖음
2. 노크 소리
3. 물소리
4. 사이렌
5. 아기 울음
6. 초인종

## 1. 환경 설정

In [ ]:
!pip install tensorflow tensorflow_hub resampy soundfile pydub matplotlib seaborn scikit-learn

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np
import os
import csv
import resampy
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from pydub import AudioSegment
import io
import random

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## 2. Google Drive 마운트 & 데이터 업로드

**사용법:** `소리 정리` 폴더를 Google Drive에 업로드한 뒤 실행하세요.

Google Drive 구조:
```
내 드라이브/
  소리 정리/
    개 짖음/
    노크 소리/
    물소리/
    사이렌/
    아기 울음/
    초인종/
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 데이터 경로 설정
DATA_DIR = '/content/drive/MyDrive/소리 정리'

# 카테고리 확인
CATEGORIES = sorted(os.listdir(DATA_DIR))
NUM_CLASSES = len(CATEGORIES)

print(f"카테고리 ({NUM_CLASSES}개): {CATEGORIES}")
for cat in CATEGORIES:
    cat_path = os.path.join(DATA_DIR, cat)
    files = [f for f in os.listdir(cat_path) if f.endswith(('.wav', '.mp3', '.flac'))]
    print(f"  {cat}: {len(files)}개")

## 3. YAMNet 모델 로드

In [ ]:
print("YAMNet 모델 로드 중...")
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')
print("YAMNet 모델 로드 완료!")

## 4. 오디오 로드 & 전처리 함수

In [ ]:
TARGET_SR = 16000  # YAMNet 요구 샘플레이트

def load_audio(file_path):
    """wav/mp3 파일을 16kHz mono float32로 로드"""
    ext = os.path.splitext(file_path)[1].lower()

    if ext == '.mp3':
        # mp3 -> wav 변환
        audio = AudioSegment.from_mp3(file_path)
        audio = audio.set_channels(1).set_frame_rate(TARGET_SR)
        samples = np.array(audio.get_array_of_samples(), dtype=np.float32) / 32768.0
        return samples
    else:
        # wav, flac 등
        wav_data, sample_rate = sf.read(file_path, dtype='float32')
        if len(wav_data.shape) > 1:
            wav_data = wav_data.mean(axis=1)  # stereo -> mono
        if sample_rate != TARGET_SR:
            wav_data = resampy.resample(wav_data, sample_rate, TARGET_SR)
        return wav_data

# 테스트
test_cat = CATEGORIES[0]
test_dir = os.path.join(DATA_DIR, test_cat)
test_file = [f for f in os.listdir(test_dir) if f.endswith(('.wav', '.mp3'))][0]
test_audio = load_audio(os.path.join(test_dir, test_file))
print(f"테스트 로드: {test_cat}/{test_file}")
print(f"  길이: {len(test_audio)/TARGET_SR:.2f}초, shape: {test_audio.shape}")

## 5. 데이터 증강 (Data Augmentation)
46개 샘플을 수백 개로 증강하여 학습 효과 향상

In [ ]:
def augment_audio(waveform, sr=TARGET_SR):
    """하나의 오디오에서 여러 증강 버전 생성"""
    augmented = []

    # 1. 원본
    augmented.append(waveform.copy())

    # 2. 가우시안 노이즈 추가 (약한)
    noise = np.random.normal(0, 0.005, len(waveform)).astype(np.float32)
    augmented.append(waveform + noise)

    # 3. 가우시안 노이즈 추가 (강한)
    noise = np.random.normal(0, 0.015, len(waveform)).astype(np.float32)
    augmented.append(waveform + noise)

    # 4. 볼륨 변경 (작게)
    augmented.append(waveform * 0.6)

    # 5. 볼륨 변경 (크게)
    augmented.append(np.clip(waveform * 1.4, -1.0, 1.0))

    # 6. Time Shift (앞뒤로 밀기)
    shift = int(sr * 0.2)  # 0.2초
    shifted = np.roll(waveform, shift)
    augmented.append(shifted)

    # 7. Time Shift (반대 방향)
    shifted = np.roll(waveform, -shift)
    augmented.append(shifted)

    # 8. Pitch Shift (속도 변경으로 근사) - 빠르게
    fast = resampy.resample(waveform, sr, int(sr * 1.1))
    if len(fast) > len(waveform):
        fast = fast[:len(waveform)]
    else:
        fast = np.pad(fast, (0, len(waveform) - len(fast)))
    augmented.append(fast)

    # 9. Pitch Shift - 느리게
    slow = resampy.resample(waveform, sr, int(sr * 0.9))
    if len(slow) > len(waveform):
        slow = slow[:len(waveform)]
    else:
        slow = np.pad(slow, (0, len(waveform) - len(slow)))
    augmented.append(slow)

    # 10. 랜덤 구간 마스킹
    masked = waveform.copy()
    mask_len = int(sr * 0.1)  # 0.1초 구간
    mask_start = random.randint(0, max(0, len(masked) - mask_len))
    masked[mask_start:mask_start + mask_len] = 0
    augmented.append(masked)

    return augmented

print(f"증강 함수 준비 완료! (원본 1개 -> {len(augment_audio(np.zeros(16000)))}개)")

## 6. 전체 데이터 로드 & 임베딩 추출

In [ ]:
def extract_embedding(waveform):
    """YAMNet에서 1024차원 임베딩 벡터 추출"""
    scores, embeddings, spectrogram = yamnet_model(waveform)
    # 여러 프레임의 임베딩을 평균내어 하나의 벡터로
    return embeddings.numpy().mean(axis=0)

# 전체 데이터 로드 + 증강 + 임베딩 추출
all_embeddings = []
all_labels = []
file_info = []  # 디버깅용

for label_idx, category in enumerate(CATEGORIES):
    cat_path = os.path.join(DATA_DIR, category)
    audio_files = [f for f in os.listdir(cat_path) if f.endswith(('.wav', '.mp3', '.flac'))]

    print(f"\n[{label_idx+1}/{NUM_CLASSES}] {category} ({len(audio_files)}개 원본)")

    cat_count = 0
    for fname in audio_files:
        fpath = os.path.join(cat_path, fname)
        try:
            waveform = load_audio(fpath)

            # 너무 짧은 오디오 패딩 (최소 1초)
            if len(waveform) < TARGET_SR:
                waveform = np.pad(waveform, (0, TARGET_SR - len(waveform)))

            # 데이터 증강
            augmented_list = augment_audio(waveform)

            for aug_waveform in augmented_list:
                embedding = extract_embedding(aug_waveform)
                all_embeddings.append(embedding)
                all_labels.append(label_idx)
                cat_count += 1

        except Exception as e:
            print(f"  [오류] {fname}: {e}")

    print(f"  -> 증강 후 {cat_count}개 샘플")

X = np.array(all_embeddings)
y = np.array(all_labels)

print(f"\n===== 데이터 준비 완료 =====")
print(f"전체 샘플 수: {len(X)}")
print(f"임베딩 차원: {X.shape[1]}")
print(f"클래스별 분포:")
for i, cat in enumerate(CATEGORIES):
    print(f"  {cat}: {np.sum(y == i)}개")

## 7. 학습/검증 데이터 분리

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"학습 데이터: {X_train.shape[0]}개")
print(f"검증 데이터: {X_test.shape[0]}개")

## 8. 분류 모델 구성 & 학습
YAMNet 임베딩(1024차원) 위에 Dense 레이어 추가

In [ ]:
# 모델 구성
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1024,)),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# 학습
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=16,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_accuracy',
            patience=15,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5
        )
    ]
)

print(f"\n최종 검증 정확도: {max(history.history['val_accuracy']):.4f}")

## 9. 학습 결과 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 정확도 그래프
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# 손실 그래프
axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 10. 혼동 행렬 (Confusion Matrix) & 분류 리포트

In [ ]:
# 예측
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# 혼동 행렬
cm = confusion_matrix(y_test, y_pred_classes)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CATEGORIES, yticklabels=CATEGORIES)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# 분류 리포트
print("\n===== Classification Report =====")
print(classification_report(y_test, y_pred_classes, target_names=CATEGORIES))

## 11. 모델 저장

In [ ]:
# Keras 모델 저장
SAVE_DIR = '/content/drive/MyDrive/Hearo_model'
os.makedirs(SAVE_DIR, exist_ok=True)

model.save(os.path.join(SAVE_DIR, 'hearo_classifier.keras'))

# 카테고리 이름 저장
with open(os.path.join(SAVE_DIR, 'categories.txt'), 'w', encoding='utf-8') as f:
    for cat in CATEGORIES:
        f.write(cat + '\n')

print(f"모델 저장 완료: {SAVE_DIR}")
print(f"  - hearo_classifier.keras")
print(f"  - categories.txt")

## 12. TFLite 변환 (Raspberry Pi 배포용)

In [ ]:
# TFLite 변환
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = os.path.join(SAVE_DIR, 'hearo_classifier.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f"TFLite 모델 저장: {tflite_path}")
print(f"모델 크기: {len(tflite_model) / 1024:.1f} KB")

## 13. 새 소리 테스트 (파인튜닝된 모델)

In [ ]:
def predict_sound(file_path):
    """파인튜닝된 모델로 소리 분류"""
    waveform = load_audio(file_path)
    if len(waveform) < TARGET_SR:
        waveform = np.pad(waveform, (0, TARGET_SR - len(waveform)))

    embedding = extract_embedding(waveform)
    embedding = embedding.reshape(1, -1)

    prediction = model.predict(embedding, verbose=0)[0]

    print(f"\n파일: {os.path.basename(file_path)}")
    print(f"길이: {len(waveform)/TARGET_SR:.1f}초")
    print(f"{'='*40}")

    sorted_idx = np.argsort(prediction)[::-1]
    for idx in sorted_idx:
        bar = '#' * int(prediction[idx] * 30)
        marker = ' <<' if idx == sorted_idx[0] else ''
        print(f"  {CATEGORIES[idx]:<10s} {prediction[idx]*100:5.1f}% {bar}{marker}")

    return CATEGORIES[sorted_idx[0]], prediction[sorted_idx[0]]

print("predict_sound() 함수 준비 완료!")
print("사용법: predict_sound('/path/to/audio.wav')")

In [ ]:
# 각 카테고리에서 1개씩 테스트
print("===== 카테고리별 샘플 테스트 =====")

results = []
for category in CATEGORIES:
    cat_path = os.path.join(DATA_DIR, category)
    audio_files = [f for f in os.listdir(cat_path) if f.endswith(('.wav', '.mp3', '.flac'))]
    if audio_files:
        test_file = os.path.join(cat_path, audio_files[0])
        predicted, confidence = predict_sound(test_file)
        correct = '  O' if predicted == category else '  X'
        results.append((category, predicted, confidence, correct))

print(f"\n\n===== 테스트 요약 =====")
correct_count = sum(1 for r in results if r[3].strip() == 'O')
print(f"정확도: {correct_count}/{len(results)} ({correct_count/len(results)*100:.0f}%)")
for actual, predicted, conf, mark in results:
    print(f"  {mark} {actual} -> {predicted} ({conf*100:.1f}%)")

## 14. 파일 업로드로 직접 테스트

In [ ]:
from google.colab import files
from IPython.display import Audio, display

print("소리 파일을 업로드하세요 (wav, mp3)")
uploaded = files.upload()

for filename in uploaded:
    waveform = load_audio(filename)
    display(Audio(waveform, rate=TARGET_SR))
    predicted, confidence = predict_sound(filename)
    print(f"\n  -> 결과: {predicted} ({confidence*100:.1f}%)\n")